# Norvale: watch people work in three systems
Run all cells to inspect the completed January–April 2027 company. These cells only read saved exports. They never start services, generate activity, or call a model.

In [1]:
from pathlib import Path
import json, html
from IPython.display import display, HTML
EXPORT = Path('../../data/observable')
records = json.loads((EXPORT / 'native-records.json').read_text())
tickets = json.loads((EXPORT / 'tickets.json').read_text())
mail = json.loads((EXPORT / 'mail.json').read_text())
def rows(kind): return list(records.get(kind, {}).values())
def show(data):
    columns = list(data[0]) if data else []
    esc = lambda value: html.escape(str(value))
    display(HTML('<table><tr>' + ''.join('<th>'+esc(c)+'</th>' for c in columns) + '</tr>' + ''.join('<tr>'+''.join('<td>'+esc(r[c])+'</td>' for c in columns)+'</tr>' for r in data)+'</table>'))
show([{'Software': 'ERPNext', 'Storage': 'MariaDB', 'Saved records': sum(map(len, records.values()))},
      {'Software': 'Gitea', 'Storage': 'Separate SQLite', 'Saved records': len(tickets)},
      {'Software': 'Email', 'Storage': 'Mail files / SMTP + IMAP', 'Saved records': len(mail)}])

Software,Storage,Saved records
ERPNext,MariaDB,3859
Gitea,Separate SQLite,15
Email,Mail files / SMTP + IMAP,119


**What you see:** real saved records from three applications. Manufacturing and customer sales are inside ERPNext. There is no separate MES or CRM application in this run.

In [2]:
show([{'Person': r.get('employee_name'), 'ERP user': r.get('user_id'), 'Reports to': r.get('reports_to')} for r in rows('Employee')[:6]])

Person,ERP user,Reports to
Henrik Lindholm,ceo@norvale.test,None
Claire Moreau,cfo@norvale.test,HR-EMP-00001
Dirk Van Der Berg,coo@norvale.test,HR-EMP-00001
Sofia Alvarez,cco@norvale.test,HR-EMP-00001
Tomas Lindqvist,plant_mgr@norvale.test,HR-EMP-00003
Vikram Desai,scm_mgr@norvale.test,HR-EMP-00003


**People:** these are seeded employees linked to software users. Role permissions control the ERP records each role can read or change. The reporting link is an Employee ID.

In [3]:
# Choose another order here and rerun this cell to follow its stored links.
ORDER = 'SAL-ORD-2027-00001'
order = records['Sales Order'][ORDER]
work = next((r for r in rows('Work Order') if r.get('sales_order') == ORDER), None)
card = next((r for r in rows('Job Card') if work and r.get('work_order') == work['name']), None)
delivery = next((r for r in rows('Delivery Note') if any(i.get('against_sales_order') == ORDER for i in r.get('items', []))), None)
invoice = next((r for r in rows('Sales Invoice') if delivery and any(i.get('delivery_note') == delivery['name'] for i in r.get('items', []))), None)
payment = next((r for r in rows('Payment Entry') if invoice and any(i.get('reference_name') == invoice['name'] for i in r.get('references', []))), None)
show([{'Role': role, 'ERP record': doc['name'] if doc else 'No linked record', 'What was recorded': value}
      for role, doc, value in [
          ('Sales', order, order.get('customer')),
          ('Planner', work, str(work.get('qty'))+' units' if work else ''),
          ('Production', card, str(card.get('total_completed_qty'))+' units; '+str(card.get('total_time_in_mins'))+' minutes' if card else ''),
          ('Warehouse', delivery, delivery.get('posting_date') if delivery else ''),
          ('Finance — invoice', invoice, invoice.get('grand_total') if invoice else ''),
          ('Finance — payment', payment, payment.get('paid_amount') if payment else '')]])

Role,ERP record,What was recorded
Sales,SAL-ORD-2027-00001,Corner Stores Group
Planner,MFG-WO-2027-00001,108.0 units
Production,PO-JOB00001,108.0 units; 86.4 minutes
Warehouse,MAT-DN-2027-00001,2027-01-02
Finance — invoice,ACC-SINV-2027-00001,264.6
Finance — payment,ACC-PAY-2027-00001,264.6


**One order:** sales records the request → planning creates a job → production records work → warehouse ships → finance bills and records payment. The joins above use native links, not matching row numbers. Missing links stay visible.

In [4]:
monthly = []
for month in ['2027-01', '2027-02', '2027-03', '2027-04']:
    ordered = sum(float(i.get('qty', 0)) for r in rows('Sales Order') if str(r.get('transaction_date', '')).startswith(month) for i in r.get('items', []))
    made = sum(float(r.get('total_completed_qty', 0)) for r in rows('Job Card') if str(r.get('posting_date', '')).startswith(month) and r.get('docstatus') == 1)
    shipped = sum(float(i.get('qty', 0)) for r in rows('Delivery Note') if str(r.get('posting_date', '')).startswith(month) and r.get('docstatus') == 1 for i in r.get('items', []))
    monthly.append({'Month': month, 'Ordered': ordered, 'Made': made, 'Shipped': shipped})
show(monthly)
maximum = max(r['Ordered'] for r in monthly)
bars = ''.join('<text x="0" y="'+str(28+idx*65)+'">'+r['Month']+'</text><rect x="95" y="'+str(12+idx*65)+'" width="'+str(r['Ordered']/maximum*400)+'" height="15" fill="#a81c27"/><rect x="95" y="'+str(30+idx*65)+'" width="'+str(r['Shipped']/maximum*400)+'" height="15" fill="#777"/>' for idx,r in enumerate(monthly))
display(HTML('<svg viewBox="0 0 560 275" role="img" aria-label="Ordered red, shipped gray, by month" style="max-width:700px;width:100%">'+bars+'</svg>'))

Month,Ordered,Made,Shipped
2027-01,12408.0,9300.0,9300.0
2027-02,11112.0,12048.0,11784.0
2027-03,14280.0,12600.0,12864.0
2027-04,12216.0,13200.0,12600.0


**Four months:** red is ordered; gray is shipped. These totals come directly from native business dates. Orders, completed work and shipments happen on different dates; open demand remains at April 30.

In [5]:
ticket = sorted(tickets, key=lambda t: t['number'])[0]
show([{'Software': 'Gitea', 'Ticket': ticket['number'], 'Title': ticket['title'], 'State now': ticket['state']}])
# Strip private deduplication comments; display the business description and resolution.
import re
display(HTML('<pre style="white-space:pre-wrap">'+html.escape(re.sub(r'<!--.*?-->', '', ticket['body'], flags=re.S).replace('\\n','\n').strip())+'</pre>'))
for comment in ticket.get('comments', []):
    print(comment['body'])

Software,Ticket,Title,State now
Gitea,2,Filler preventive service,closed


Technician inspection and restart test completed 2027-01-14. ERP task mbr2gn5vi5


**Maintenance:** a separate ticket service records the equipment problem and resolution. Its text carries the simulated dates and the ERP task ID; server timestamps use the real clock.

In [6]:
message = mail[0]['message']
show([{'Software': 'Email', 'From': message['from'], 'To': message['to'], 'Subject': message['subject'], 'Business date': message['date']}])
print(message['body'].strip())
po = re.search(r'PUR-ORD-\d{4}-\d+', message['body'])
if po and po.group() in records.get('Purchase Order', {}):
    purchase = records['Purchase Order'][po.group()]
    show([{'ERP purchase': purchase['name'], 'Supplier': purchase['supplier'], 'Total': purchase['grand_total'], 'Status at cutoff': purchase['status']}])

Software,From,To,Subject,Business date
Email,priya.nair@norvale.test,vikram.desai@norvale.test,[Ref 2726f5259ac1dd92] Replenishment approval requested,"Fri, 01 Jan 2027 00:00:00 +0000"


Native purchase PUR-ORD-2027-00001 replenishes released inventory below lead-time plus safety coverage.


ERP purchase,Supplier,Total,Status at cutoff
PUR-ORD-2027-00001,Aromatica Fine Fragrances BV,222.0,Completed


**Buyer → manager:** this delivered email asks for approval and names an ERP purchase. Selected judgment cases use JEV to decide whether to request an LLM decision. Routine bookings run in code. JEV coverage was corrected February 4; this trajectory recorded no standing-policy executions.

**Scope:** these records support inspection of linked events. An extraction benchmark has not been run; private runtime references are excluded from this release.